In [ ]:
# 이미지 처리와 시각화에 필요한 OpenCV 및 Matplotlib를 설치합니다.
!pip install opencv-python==4.11.0.86 matplotlib==3.9.4

In [ ]:
# Python의 모듈 검색 경로를 다루기 위해 sys를 불러옵니다.
import sys
# RetinaFace 프로젝트 폴더를 모듈 검색 경로에 추가합니다.
sys.path.append(r'c:\ai_project01\Pytorch_Retinaface')

In [ ]:
# PyTorch의 텐서와 GPU 연산 기능을 불러옵니다.
import torch
# 이미지 읽기, 전처리, 화면 표시를 위해 OpenCV를 불러옵니다.
import cv2
# 배열 계산과 좌표 처리를 위해 NumPy를 불러옵니다.
import numpy as np

# RetinaFace 모델 클래스를 불러옵니다.
from models.retinaface import RetinaFace
# ResNet-50용 RetinaFace 설정을 불러옵니다.
from data import cfg_re50
# 기본(anchor) 박스를 생성하는 클래스를 불러옵니다.
from layers.functions.prior_box import PriorBox
# 겹치는 탐지 상자를 제거하는 NMS 함수를 불러옵니다.
from utils.nms.py_cpu_nms import py_cpu_nms
# 모델의 상대 좌표를 실제 얼굴 상자 좌표로 변환하는 함수를 불러옵니다.
from utils.box_utils import decode

In [ ]:
# CUDA GPU를 사용할 수 있는지 확인합니다.
torch.cuda.is_available()

In [ ]:
# 모델과 입력 데이터를 실행할 장치로 CUDA GPU를 지정합니다.
device = "cuda"

In [ ]:
# RetinaFace 모델의 ResNet-50 설정을 선택합니다.
cfg = cfg_re50
# 테스트 단계의 RetinaFace 모델을 만들고 지정한 장치로 이동합니다.
net = RetinaFace(cfg=cfg, phase='test').to(device)

In [ ]:
# 학습된 RetinaFace 가중치 파일의 위치를 지정합니다.
pretrained_path = r'c:\ai_project01\Pytorch_Retinaface\weights\Resnet50_Final.pth'
# 가중치를 읽고 현재 실행 장치에 맞게 배치합니다.
state_dict = torch.load(pretrained_path, map_location=device)

In [ ]:
# 불러온 가중치의 키와 값을 확인합니다.
state_dict

In [ ]:
# module. 접두사가 제거된 새 가중치 딕셔너리를 준비합니다.
new_state_dic = {}
# 저장된 모든 가중치 이름과 텐서를 하나씩 확인합니다.
for k, v in state_dict.items():
    # 여러 GPU 학습으로 저장된 가중치인지 확인합니다.
    if k.startswith("module."):
        # module. 접두사를 제거해 단일 GPU 모델 이름에 맞춥니다.
        new_state_dic[k[7:]] = v
    else:
        # 접두사가 없는 가중치는 이름을 그대로 사용합니다.
        print("이름이 module. 으로 시작 안해")
        new_state_dic[k] = v

In [ ]:
# 준비한 학습 가중치를 RetinaFace 모델에 적용합니다.
net.load_state_dict(new_state_dic, strict=True)

In [ ]:
# 모델을 평가 모드로 전환해 추론용 동작을 사용합니다.
net.eval()

In [ ]:
# 탐지할 이미지 파일의 경로를 지정합니다.
image_path = r'C:\ai_project01\mask_images\no_mask\no_mask_00000.png'

# 이미지를 컬러 형식으로 읽습니다.
img_raw = cv2.imread(image_path, cv2.IMREAD_COLOR)

# 모델 입력 전처리를 위해 이미지 자료형을 float32로 변환합니다.
img = np.float32(img_raw)

In [ ]:
# 이미지의 높이와 너비를 확인합니다.
im_height, im_width, _ = img.shape

# RetinaFace 학습 방식에 맞게 BGR 채널별 평균값을 뺍니다.
img -= (104, 117, 123)

# 이미지 차원을 높이-너비-채널에서 채널-높이-너비 순서로 바꿉니다.
img = img.transpose(2, 0, 1)

# 배치 차원을 추가하고 이미지를 CUDA GPU로 이동합니다.
img = torch.from_numpy(img).unsqueeze(0).to("cuda")

In [ ]:
# 입력 이미지를 모델에 넣어 위치, 신뢰도, 랜드마크 예측값을 계산합니다.
loc, conf, landms = net(img)

In [ ]:
# 이미지 크기에 맞는 기본(anchor) 박스 생성기를 만듭니다.
priorbox = PriorBox(cfg, image_size=(im_height, im_width))

In [ ]:
# 기본 박스를 생성하고 모델과 같은 CUDA 장치로 이동합니다.
priors = priorbox.forward().to("cuda")

In [ ]:
# 기본 박스 텐서에서 실제 데이터 부분을 가져옵니다.
prior_data = priors.data

In [ ]:
# 기본 박스 텐서의 크기를 확인합니다.
prior_data.shape

In [ ]:
# 모델이 예측한 위치 데이터의 크기를 확인합니다.
loc.data.squeeze(0).shape

In [ ]:
# 모델의 상대 위치 예측값을 기본 박스 기준의 얼굴 상자 좌표로 복원합니다.
boxes = decode(loc.data.squeeze(0), prior_data, cfg["variance"])

In [ ]:
# 정규화된 좌표를 이미지 픽셀 좌표로 바꾸기 위한 크기 벡터를 만듭니다.
scale = torch.Tensor([im_width, im_height, im_width, im_height]).to("cuda")

In [ ]:
# 크기 변환 전 얼굴 상자 좌표를 확인합니다.
boxes

In [ ]:
# 좌표 변환에 사용할 이미지 크기 벡터를 확인합니다.
scale

In [ ]:
# 얼굴 상자 좌표를 이미지의 실제 픽셀 단위로 변환합니다.
boxes = boxes * scale

In [ ]:
# 각 기본 박스에 대한 얼굴 탐지 신뢰도만 CPU NumPy 배열로 가져옵니다.
scores = conf.squeeze(0).data.cpu().numpy()[:, 1]

In [ ]:
# 계산된 모든 탐지 신뢰도 점수를 확인합니다.
scores

In [ ]:
# 얼굴로 판단할 최소 신뢰도 기준을 0.5로 설정합니다.
confience_threshold = 0.5

In [ ]:
# 각 탐지 결과가 기준 신뢰도보다 높은지 True 또는 False로 확인합니다.
scores > confience_threshold

In [ ]:
# 기준 신뢰도를 넘은 탐지 결과의 인덱스만 선택합니다.
top_indices = np.where(scores > confience_threshold)[0]

In [ ]:
# 신뢰도가 낮은 얼굴 상자는 제외하고 상자 좌표를 필터링합니다.
boxes = boxes[top_indices]

In [ ]:
# 선택된 얼굴 상자에 해당하는 신뢰도만 남깁니다.
scores = scores[top_indices]

In [ ]:
# 필터링된 얼굴 상자를 NumPy 배열로 확인합니다.
boxes.cpu().numpy()

In [ ]:
# 신뢰도 배열을 열 벡터 형태로 바꿔 상자 데이터와 결합할 준비를 합니다.
scores[:, np.newaxis]

In [ ]:
# 얼굴 상자 좌표와 신뢰도 점수를 하나의 NumPy 배열로 합칩니다.
dets = np.hstack((boxes.cpu().numpy(), scores[:, np.newaxis])).astype(np.float32, copy=False)

In [ ]:
# 좌표와 신뢰도가 합쳐진 전체 탐지 결과를 확인합니다.
dets

In [ ]:
# IoU 기준 0.3을 사용해 서로 겹치는 중복 상자의 인덱스를 구합니다.
keep = py_cpu_nms(dets, 0.3)

In [ ]:
# NMS를 통과한 최종 탐지 상자와 신뢰도를 확인합니다.
dets[keep, :]

In [ ]:
# 최종 탐지 결과를 하나씩 확인합니다.
for b in dets:
    # 신뢰도가 기준보다 낮은 결과는 화면에 표시하지 않습니다.
    if b[4] < confience_threshold:
        continue
    # 신뢰도 점수를 소수 둘째 자리의 문자열로 만듭니다.
    text = "{:.2f}".format(b[4])
    # 상자 좌표를 그리기 위해 정수로 변환합니다.
    b = list(map(int, b))
    # 이미지에 얼굴 탐지 상자를 초록색 사각형으로 그립니다.
    cv2.rectangle(img_raw, (b[0], b[1]), (b[2], b[3]), (0, 255, 0), 2)
    # 신뢰도 문자를 표시할 위치를 계산합니다.
    cx, cy = b[0], b[1] + 12
    # 얼굴 상자 근처에 신뢰도 점수를 그립니다.
    cv2.putText(img_raw, text, (cx, cy), cv2.FONT_HERSHEY_DUPLEX, 0.5, (255, 255, 255))
    # 결과 이미지를 창에 표시합니다.
    cv2.imshow('RetinaFace Pytorch Detection', img_raw)
    # 키를 누를 때까지 결과 창을 유지합니다.
    cv2.waitKey(0)
    # 결과 창을 닫습니다.
    cv2.destroyAllWindows()